# Flight Price Prediction - EDA

Kaggle competition: flight-price-prediction

Target is `price` (USD). Metric is R2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, skew
from sklearn.metrics import r2_score

SEED = 42

In [ ]:
path = "/kaggle/input/competitions/flight-price-prediction/"
train = pd.read_csv(path + "train.csv")
test = pd.read_csv(path + "test.csv")

print(train.shape, test.shape)
train.head()

In [ ]:
train.info()
print(train.isna().sum().sum(), test.isna().sum().sum())

cols = [c for c in train.columns if c not in ["id", "price"]]
print("duplicate rows:", train.duplicated(subset=cols + ["price"]).sum())

In [ ]:
cat_cols = ["airline", "source_city", "destination_city", "departure_time",
            "arrival_time", "stops", "flight_class"]

for c in cat_cols:
    unseen = set(test[c]) - set(train[c])
    print(c, train[c].nunique(), "unseen in test:", unseen if unseen else "none")
    print(train[c].value_counts().to_string(), "\n")

print("flights:", train["flight"].nunique(), "| test flights not in train:", len(set(test["flight"]) - set(train["flight"])))

train["route"] = train["source_city"] + " -> " + train["destination_city"]
test["route"] = test["source_city"] + " -> " + test["destination_city"]
print("routes:", train["route"].nunique())

In [ ]:
print(train[["duration", "days_left", "price"]].describe().T.round(2))
print(test[["duration", "days_left"]].describe().T.round(2))

## Target (price)

In [ ]:
p = train["price"]
print("median", p.median(), "| max", p.max())
print(p.quantile([0.01, 0.25, 0.5, 0.75, 0.9, 0.99]))
print("skew raw:", round(skew(p), 2), "| skew log:", round(skew(np.log1p(p)), 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(p, bins=80, ax=axes[0])
axes[0].set_title("price")
sns.histplot(np.log1p(p), bins=80, ax=axes[1], color="darkorange")
axes[1].set_title("log1p(price)")
plt.show()

# very skewed -> use log price for training

In [ ]:
# zero prices
zero = train[train["price"] == 0]
print("zero price rows:", len(zero))
print(zero.groupby(["airline", "route", "stops", "flight_class"]).size().sort_values(ascending=False))

In [ ]:
# expensive tickets
print(train.sort_values("price", ascending=False)[["id", "airline", "flight", "route", "flight_class", "days_left", "price"]].head(15))

hi = train[train["price"] > 1000]
print("\nprice > 1000:", len(hi))
print(hi["flight_class"].value_counts())
print(hi.groupby(["flight", "route", "flight_class"]).size().sort_values(ascending=False).head(10))
print("\nrepeated prices:")
print(hi["price"].value_counts().head(10))

In [ ]:
# how much do the biggest prices matter for R2
d = (p - p.mean()) ** 2
d = d.sort_values(ascending=False)
print("top row:", round(d.iloc[0] / d.sum() * 100, 1), "%")
print("top 10 rows:", round(d.iloc[:10].sum() / d.sum() * 100, 1), "%")
print("top 1%:", round(d.iloc[:len(p) // 100].sum() / d.sum() * 100, 1), "%")

In [ ]:
# outliers (IQR)
for c in ["price", "duration", "days_left"]:
    q1, q3 = train[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((train[c] < q1 - 1.5 * iqr) | (train[c] > q3 + 1.5 * iqr)).sum()
    print(c, n, round(n / len(train) * 100, 2), "%")

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, c in zip(axes, ["price", "duration", "days_left"]):
    sns.boxplot(x=train[c], ax=ax, color="lightcoral")
plt.show()

## Features vs price

In [ ]:
plot_cols = ["flight_class", "airline", "stops", "departure_time", "arrival_time", "source_city", "destination_city"]

fig, axes = plt.subplots(4, 2, figsize=(16, 18))
axes = axes.flatten()
for ax, c in zip(axes, plot_cols):
    order = train.groupby(c)["price"].median().sort_values().index
    sns.boxplot(data=train, x=c, y="price", order=order, ax=ax, showfliers=False)
    ax.set_yscale("log")
    ax.set_title(c)
    ax.tick_params(axis="x", rotation=30)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

for c in plot_cols:
    print(train.groupby(c)["price"].agg(["count", "median", "mean"]).round(1).sort_values("median", ascending=False), "\n")

In [ ]:
print(train.pivot_table(index="airline", columns="flight_class", values="price", aggfunc="median"))
print(train.pivot_table(index="stops", columns="flight_class", values="price", aggfunc="median"))

In [ ]:
# days_left
med = train.groupby(["days_left", "flight_class"])["price"].median().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.lineplot(data=med, x="days_left", y="price", hue="flight_class", marker="o", ax=axes[0])
axes[0].set_title("median price vs days_left")
cnt = train["days_left"].value_counts().sort_index()
axes[1].bar(cnt.index, cnt.values)
axes[1].set_title("rows per days_left")
plt.show()

eco = train[train["flight_class"] == "Economy"]
print(eco.groupby(pd.cut(eco["days_left"], [0, 3, 7, 14, 21, 30, 50]))["price"].median())
print("spearman:", round(spearmanr(train["days_left"], train["price"])[0], 3))

In [ ]:
# duration
sns.scatterplot(data=train.sample(5000, random_state=SEED), x="duration", y="price", hue="flight_class", s=10, alpha=0.4)
plt.yscale("log")
plt.show()
print("spearman:", round(spearmanr(train["duration"], train["price"])[0], 3))

In [ ]:
# correlation with log price
stops_map = {"zero": 0, "one": 1, "two_or_more": 2}
time_map = {"Early_Morning": 0, "Morning": 1, "Afternoon": 2, "Evening": 3, "Night": 4, "Late_Night": 5}

enc = pd.DataFrame({
    "duration": train["duration"],
    "days_left": train["days_left"],
    "stops": train["stops"].map(stops_map),
    "dep": train["departure_time"].map(time_map),
    "arr": train["arrival_time"].map(time_map),
    "business": (train["flight_class"] == "Business").astype(int),
    "log_price": np.log1p(train["price"]),
})
corr = enc.corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.show()

In [ ]:
# how much of log price each feature explains on its own (group mean R2)
lp = np.log1p(train["price"])
train["dl_bucket"] = pd.cut(train["days_left"], [0, 3, 7, 14, 21, 30, 50])
train["dur_bucket"] = pd.qcut(train["duration"], 10, duplicates="drop")

feats = ["flight", "flight_class", "route", "dur_bucket", "stops", "dl_bucket", "airline",
         "source_city", "destination_city", "departure_time", "arrival_time"]
sst = ((lp - lp.mean()) ** 2).sum()
eta = {}
for c in feats:
    eta[c] = 1 - ((lp - lp.groupby(train[c]).transform("mean")) ** 2).sum() / sst
eta = pd.Series(eta).sort_values(ascending=False)

sns.barplot(x=eta.values, y=eta.index)
plt.title("variance of log price explained by each feature")
plt.show()
print(eta.round(3))

In [ ]:
# train vs test
for c in ["flight_class", "airline", "stops"]:
    print(pd.concat([train[c].value_counts(normalize=True), test[c].value_counts(normalize=True)], axis=1, keys=["train", "test"]).round(3), "\n")

## Something interesting with id
Looks like id follows the order the data was collected in. Checking it.

In [ ]:
allx = pd.concat([train.assign(is_train=1), test.assign(is_train=0)], ignore_index=True)
allx = allx.sort_values("id").reset_index(drop=True)

print("spearman(id, days_left):", round(spearmanr(allx["id"], allx["days_left"])[0], 3))

perm = np.random.default_rng(SEED).permutation(len(allx))
for c in ["days_left", "route", "flight_class", "airline"]:
    same = (allx[c].values[1:] == allx[c].values[:-1]).mean()
    rand = (allx[c].values[perm][1:] == allx[c].values[perm][:-1]).mean()
    print(c, "next row same:", round(same * 100, 1), "% | random:", round(rand * 100, 1), "%")

plt.figure(figsize=(12, 4))
plt.scatter(allx["id"], allx["days_left"], s=1)
plt.xlabel("id")
plt.ylabel("days_left")
plt.show()

In [ ]:
# blocks of rows with same days_left + route + class
# are prices sorted inside a block?
key = allx["days_left"].astype(str) + allx["route"] + allx["flight_class"]
allx["block"] = (key != key.shift()).cumsum()

rhos = []
for b, g in allx[allx["is_train"] == 1].groupby("block"):
    if len(g) >= 4 and g["price"].nunique() > 1:
        rhos.append(spearmanr(g["id"], g["price"])[0])
rhos = np.array(rhos)

print("blocks:", allx["block"].nunique())
print("median spearman(id, price) in block:", round(np.median(rhos), 3))
print("blocks with > 0.5:", round((rhos > 0.5).mean() * 100, 1), "%")

sns.histplot(rhos, bins=40)
plt.show()

In [ ]:
# just interpolate price from neighbours in the same block
tr = allx[allx["is_train"] == 1][["id", "block", "price"]].reset_index(drop=True)
pred = np.full(len(tr), np.nan)
for b, g in tr.groupby("block"):
    if len(g) < 2:
        continue
    ids = g["id"].values.astype(float)
    pr = g["price"].values
    for i in range(len(g)):
        others = np.delete(np.arange(len(g)), i)
        pred[g.index[i]] = np.interp(ids[i], ids[others], pr[others])

ok = ~np.isnan(pred)
print("R2 from interpolation only:", round(r2_score(tr["price"][ok], pred[ok]), 4))

Notes:
- price is very skewed -> log target
- 25 zero prices, will drop them from training
- prices > 1000 are almost all business and repeat a lot (fare buckets)
- flight, class and route matter the most, time of day almost nothing
- id order has a lot of info (sorted blocks), will make features from it